In [1]:
# S2-DAT-02 — Load Risk Questionnaire v1
#
# Purpose:
# Load the approved/validated FinEx Risk Questionnaire v1 definition
# into the operational PostgreSQL/Lakebase risk questionnaire tables.
#
# Source:
# data/questionnaire/risk_questionnaire_v1.json
#
# Expected:
# - 1 questionnaire version
# - 10 questions
# - 50 answer options
# - 5 willingness questions
# - 5 capacity questions
#
# This notebook is intended to be idempotent and transaction-safe.

In [2]:
import json
from datetime import datetime
from pathlib import Path

from sqlalchemy import text

In [3]:
BASE_DIR = Path.cwd().parent
QUESTIONNAIRE_PATH = BASE_DIR / "data" / "questionnaire" / "risk_questionnaire_v1.json"

if not QUESTIONNAIRE_PATH.is_file():
    raise FileNotFoundError(f"Questionnaire file not found: {QUESTIONNAIRE_PATH}")

with QUESTIONNAIRE_PATH.open(encoding="utf-8") as f:
    questionnaire_data = json.load(f)

print(f"Loaded: {QUESTIONNAIRE_PATH}")

Loaded: c:\Users\UNNATI SATPUTE\Finex\backend\data\questionnaire\risk_questionnaire_v1.json


In [4]:
from collections import Counter


class QuestionnaireValidationError(ValueError):
    """Raised when the questionnaire seed fails validation."""


def validate_questionnaire(data: dict) -> tuple[dict, list[dict]]:
    """Validate the complete questionnaire v1 seed contract."""

    if not isinstance(data, dict):
        raise QuestionnaireValidationError("Root JSON must be an object.")

    questionnaire = data.get("questionnaire")
    questions = data.get("questions")

    if not isinstance(questionnaire, dict):
        raise QuestionnaireValidationError("Missing or invalid 'questionnaire' object.")

    if not isinstance(questions, list):
        raise QuestionnaireValidationError("Missing or invalid 'questions' array.")

    required_questionnaire_fields = {
        "questionnaire_id",
        "version_label",
        "title",
        "status",
        "active_from",
        "retired_at",
    }

    missing = required_questionnaire_fields - questionnaire.keys()
    if missing:
        raise QuestionnaireValidationError(f"Missing questionnaire fields: {sorted(missing)}")

    if questionnaire["questionnaire_id"] != "risk_questionnaire_v1":
        raise QuestionnaireValidationError(
            f"Unexpected questionnaire_id: {questionnaire['questionnaire_id']!r}"
        )

    if questionnaire["version_label"] != "v1":
        raise QuestionnaireValidationError(
            f"Expected version_label='v1', got {questionnaire['version_label']!r}"
        )

    if questionnaire["status"] != "APPROVED":
        raise QuestionnaireValidationError(
            "Questionnaire must be APPROVED before loading."
        )

    if len(questions) != 10:
        raise QuestionnaireValidationError(f"Expected 10 questions, found {len(questions)}.")

    positions = []
    question_ids = []
    answer_codes = []
    dimensions = Counter()

    for index, question in enumerate(questions, start=1):
        if not isinstance(question, dict):
            raise QuestionnaireValidationError(f"Question at index {index} must be an object.")

        required_fields = {
            "position",
            "question_id",
            "dimension",
            "text",
            "response_type",
            "required",
            "answers",
        }

        missing = required_fields - question.keys()
        if missing:
            raise QuestionnaireValidationError(
                f"Question {index} missing fields: {sorted(missing)}"
            )

        position = question["position"]
        question_id = question["question_id"]
        dimension = question["dimension"]
        text = question["text"]
        answers = question["answers"]

        if position != index:
            raise QuestionnaireValidationError(
                f"Question {question_id!r} has position {position}; expected {index}."
            )

        if not isinstance(question_id, str) or not question_id.strip():
            raise QuestionnaireValidationError(f"Question {index} has an invalid question_id.")

        if dimension not in {"willingness", "capacity"}:
            raise QuestionnaireValidationError(f"{question_id}: invalid dimension {dimension!r}.")

        if not isinstance(text, str) or not text.strip():
            raise QuestionnaireValidationError(f"{question_id}: question text cannot be empty.")

        if question["response_type"] != "single_choice":
            raise QuestionnaireValidationError(
                f"{question_id}: unsupported response_type {question['response_type']!r}."
            )

        if question["required"] is not True:
            raise QuestionnaireValidationError(f"{question_id}: required must be true.")

        if not isinstance(answers, list) or len(answers) != 5:
            raise QuestionnaireValidationError(f"{question_id}: expected exactly 5 answer options.")

        positions.append(position)
        question_ids.append(question_id)
        dimensions[dimension] += 1

        question_answer_codes = []

        for answer in answers:
            answer_fields = {"code", "label", "proposed_score"}

            missing = answer_fields - answer.keys()
            if missing:
                raise QuestionnaireValidationError(
                    f"{question_id}: answer missing fields {sorted(missing)}."
                )

            code = answer["code"]
            label = answer["label"]
            score = answer["proposed_score"]

            if not isinstance(code, str) or not code.strip():
                raise QuestionnaireValidationError(f"{question_id}: answer code cannot be empty.")

            if not isinstance(label, str) or not label.strip():
                raise QuestionnaireValidationError(
                    f"{question_id}/{code}: answer label cannot be empty."
                )

            if not isinstance(score, int) or not 1 <= score <= 5:
                raise QuestionnaireValidationError(
                    f"{question_id}/{code}: proposed_score must be an integer from 1 to 5."
                )

            question_answer_codes.append(code)
            answer_codes.append(code)

        if len(set(question_answer_codes)) != 5:
            raise QuestionnaireValidationError(f"{question_id}: duplicate answer code detected.")

    if positions != list(range(1, 11)):
        raise QuestionnaireValidationError(f"Invalid question positions: {positions}")

    if len(set(question_ids)) != 10:
        raise QuestionnaireValidationError("Duplicate question_id detected.")

    if len(answer_codes) != 50:
        raise QuestionnaireValidationError(
            f"Expected 50 answer options, found {len(answer_codes)}."
        )

    if len(set(answer_codes)) != 50:
        duplicates = [code for code, count in Counter(answer_codes).items() if count > 1]
        raise QuestionnaireValidationError(f"Duplicate answer codes: {duplicates}")

    if dimensions != {"willingness": 5, "capacity": 5}:
        raise QuestionnaireValidationError(f"Unexpected dimension distribution: {dict(dimensions)}")

    return questionnaire, questions

In [5]:
questionnaire, questions = validate_questionnaire(questionnaire_data)

print("QUESTIONNAIRE VALIDATION: PASS")
print(f"Version: {questionnaire['version_label']}")
print(f"Status: {questionnaire['status']}")
print(f"Questions: {len(questions)}")
print(f"Answer options: {sum(len(q['answers']) for q in questions)}")
print(
    "Dimensions:",
    dict(Counter(q["dimension"] for q in questions)),
)

QUESTIONNAIRE VALIDATION: PASS
Version: v1
Status: APPROVED
Questions: 10
Answer options: 50
Dimensions: {'willingness': 5, 'capacity': 5}


In [6]:
import os
import sys

from dotenv import load_dotenv

BACKEND_DIR = BASE_DIR

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

load_dotenv(BACKEND_DIR / ".env")

DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise RuntimeError(
        "DATABASE_URL is not configured. Set it in backend/.env before running the loader."
    )

print("Database configuration loaded.")

Database configuration loaded.


In [7]:
from app.ports.adapters.lakebase import AsyncSessionLocal

print("Async database session factory loaded.")

Async database session factory loaded.


In [8]:
async with AsyncSessionLocal() as session:
    result = await session.execute(text("SELECT 1"))
    print("Database connection:", result.scalar_one())

Database connection: 1


In [9]:
async with AsyncSessionLocal() as session:
    result = await session.execute(
        text(
            """
            SELECT questionnaire_id, version_label, active_from, retired_at
            FROM risk_questionnaires
            WHERE version_label = :version_label
            """
        ),
        {"version_label": questionnaire["version_label"]},
    )

    existing_questionnaire = result.mappings().first()

print("Existing v1:", existing_questionnaire)

Existing v1: {'questionnaire_id': UUID('7129a8ce-b0e2-478e-bf01-8ba9e8949f55'), 'version_label': 'v1', 'active_from': None, 'retired_at': None}


In [10]:
# Load the questionnaire definition only when the requested version does
# not already exist. If it exists, leave the persisted definition untouched;
# the later integrity/idempotency checks will verify that it matches the source.

from uuid import uuid4

async with AsyncSessionLocal.begin() as session:
    existing = await session.execute(
        text(
            """
            SELECT questionnaire_id
            FROM risk_questionnaires
            WHERE version_label = :version_label
            """
        ),
        {"version_label": questionnaire["version_label"]},
    )

    existing_row = existing.mappings().first()

    if existing_row is not None:
        await session.execute(
            text(
                """
                UPDATE risk_questionnaires
                SET
                    active_from = :active_from,
                    retired_at = :retired_at
                WHERE questionnaire_id = :questionnaire_id
                """
            ),
            {
                "questionnaire_id": existing_row["questionnaire_id"],
                "active_from": (
                    datetime.fromisoformat(questionnaire["active_from"])
                    if questionnaire["active_from"]
                    else None
                ),
                "retired_at": (
                    datetime.fromisoformat(questionnaire["retired_at"])
                    if questionnaire["retired_at"]
                    else None
                ),
            },
        )

        print(
            f"Questionnaire {questionnaire['version_label']!r} already exists. "
            "Updated activation metadata."
        )
    else:
        questionnaire_uuid = uuid4()

        # Insert the questionnaire version and all of its questions in
        # one transaction so a partial seed cannot be committed.
        await session.execute(
            text(
                """
                INSERT INTO risk_questionnaires (
                    questionnaire_id,
                    version_label,
                    active_from,
                    retired_at
                )
                VALUES (
                    :questionnaire_id,
                    :version_label,
                    :active_from,
                    :retired_at
                )
                """
            ),
            {
                "questionnaire_id": questionnaire_uuid,
                "version_label": questionnaire["version_label"],
                "active_from": (
                    datetime.fromisoformat(questionnaire["active_from"])
                    if questionnaire["active_from"]
                    else None
                ),
                "retired_at": (
                    datetime.fromisoformat(questionnaire["retired_at"])
                    if questionnaire["retired_at"]
                    else None
                ),
            },
        )

        for question in questions:
            await session.execute(
                text(
                    """
                    INSERT INTO risk_questions (
                        question_id,
                        questionnaire_id,
                        position,
                        text_key,
                        response_type,
                        required,
                        dimension,
                        answer_options
                    )
                    VALUES (
                        :question_id,
                        :questionnaire_id,
                        :position,
                        :text_key,
                        :response_type,
                        :required,
                        :dimension,
                        CAST(:answer_options AS JSON)
                    )
                    """
                ),
                {
                    "question_id": uuid4(),
                    "questionnaire_id": questionnaire_uuid,
                    "position": question["position"],
                    "text_key": question["question_id"],
                    "response_type": question["response_type"],
                    "required": question["required"],
                    "dimension": question["dimension"],
                    "answer_options": json.dumps(question["answers"]),
                },
            )

        print(
            f"Loaded questionnaire {questionnaire['version_label']!r} "
            f"with {len(questions)} questions."
        )

Questionnaire 'v1' already exists. Updated activation metadata.


In [11]:
async with AsyncSessionLocal() as session:
    questionnaire_result = await session.execute(
        text(
            """
            SELECT
                questionnaire_id,
                version_label,
                active_from,
                retired_at
            FROM risk_questionnaires
            WHERE version_label = :version_label
            """
        ),
        {"version_label": questionnaire["version_label"]},
    )

    questionnaire_row = questionnaire_result.mappings().one()

    questions_result = await session.execute(
        text(
            """
            SELECT
                question_id,
                questionnaire_id,
                position,
                text_key,
                response_type,
                required,
                dimension,
                answer_options
            FROM risk_questions
            WHERE questionnaire_id = :questionnaire_id
            ORDER BY position
            """
        ),
        {"questionnaire_id": questionnaire_row["questionnaire_id"]},
    )

    question_rows = questions_result.mappings().all()

print("QUESTIONNAIRE ROWS:", 1)
print("QUESTION ROWS:", len(question_rows))
print("VERSION:", questionnaire_row["version_label"])
print("POSITIONS:", [row["position"] for row in question_rows])
print(
    "DIMENSIONS:",
    {
        "willingness": sum(row["dimension"] == "willingness" for row in question_rows),
        "capacity": sum(row["dimension"] == "capacity" for row in question_rows),
    },
)

QUESTIONNAIRE ROWS: 1
QUESTION ROWS: 10
VERSION: v1
POSITIONS: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
DIMENSIONS: {'willingness': 5, 'capacity': 5}


In [12]:
total_answers = sum(len(row["answer_options"]) for row in question_rows)

all_scores = [answer["proposed_score"] for row in question_rows for answer in row["answer_options"]]

all_codes = [answer["code"] for row in question_rows for answer in row["answer_options"]]

if total_answers != 50:
    raise RuntimeError(f"Expected 50 answer options, found {total_answers}.")

if len(set(all_codes)) != 50:
    raise RuntimeError("Duplicate answer codes found in database.")

if sorted(set(all_scores)) != [1, 2, 3, 4, 5]:
    raise RuntimeError(f"Unexpected scoring weights: {sorted(set(all_scores))}")

print("SCORING VERIFICATION: PASS")
print("Answer options:", total_answers)
print("Unique answer codes:", len(set(all_codes)))
print("Scoring weights:", sorted(set(all_scores)))

SCORING VERIFICATION: PASS
Answer options: 50
Unique answer codes: 50
Scoring weights: [1, 2, 3, 4, 5]


In [13]:
async with AsyncSessionLocal() as session:
    result = await session.execute(text("SELECT COUNT(*) FROM risk_assessments"))
    assessment_count = result.scalar_one()

print("Existing assessments:", assessment_count)

Existing assessments: 0


In [14]:
async def verify_existing_v1() -> None:
    """Verify an existing v1 seed matches the source definition."""

    async with AsyncSessionLocal() as session:
        questionnaire_result = await session.execute(
            text(
                """
                SELECT questionnaire_id, version_label
                FROM risk_questionnaires
                WHERE version_label = :version_label
                """
            ),
            {"version_label": questionnaire["version_label"]},
        )

        questionnaire_row = questionnaire_result.mappings().first()

        if questionnaire_row is None:
            print("v1 does not exist yet; initial load is required.")
            return

        question_result = await session.execute(
            text(
                """
                SELECT
                    position,
                    text_key,
                    response_type,
                    required,
                    dimension,
                    answer_options
                FROM risk_questions
                WHERE questionnaire_id = :questionnaire_id
                ORDER BY position
                """
            ),
            {"questionnaire_id": questionnaire_row["questionnaire_id"]},
        )

        existing_questions = question_result.mappings().all()

    if len(existing_questions) != len(questions):
        raise RuntimeError("Existing v1 question count does not match source.")

    for source, existing in zip(questions, existing_questions):
        if source["position"] != existing["position"]:
            raise RuntimeError(f"Position mismatch for {source['question_id']}.")

        if source["question_id"] != existing["text_key"]:
            raise RuntimeError(f"Question key mismatch for {source['question_id']}.")

        if source["response_type"] != existing["response_type"]:
            raise RuntimeError(f"Response type mismatch for {source['question_id']}.")

        if source["required"] != existing["required"]:
            raise RuntimeError(f"Required flag mismatch for {source['question_id']}.")

        if source["dimension"] != existing["dimension"]:
            raise RuntimeError(f"Dimension mismatch for {source['question_id']}.")

        if source["answers"] != existing["answer_options"]:
            raise RuntimeError(f"Answer/scoring metadata mismatch for {source['question_id']}.")

    print("IDEMPOTENCY CHECK: PASS")
    print("Existing v1 matches the source questionnaire exactly.")

In [15]:
await verify_existing_v1()

IDEMPOTENCY CHECK: PASS
Existing v1 matches the source questionnaire exactly.


In [16]:
# Validate structural and business-rule constraints for the loaded
# questionnaire definition. These checks ensure the database contains
# the expected number of questions and that required questionnaire
# attributes were preserved during loading.

async with AsyncSessionLocal() as session:
    result = await session.execute(
        text(
            """
            SELECT
                COUNT(*) AS total_questions,
                COUNT(DISTINCT question_id) AS unique_question_ids,
                COUNT(DISTINCT position) AS unique_positions,
                COUNT(*) FILTER (
                    WHERE dimension = 'willingness'
                ) AS willingness_questions,
                COUNT(*) FILTER (
                    WHERE dimension = 'capacity'
                ) AS capacity_questions,
                COUNT(*) FILTER (
                    WHERE response_type = 'single_choice'
                ) AS single_choice_questions,
                COUNT(*) FILTER (
                    WHERE required = TRUE
                ) AS required_questions
            FROM risk_questions rq
            JOIN risk_questionnaires rqs
                ON rq.questionnaire_id = rqs.questionnaire_id
            WHERE rqs.version_label = :version_label
            """
        ),
        {"version_label": questionnaire["version_label"]},
    )

    dq = result.mappings().one()


# Expected values come from the validated v1 questionnaire contract.
expected = {
    "total_questions": 10,
    "unique_question_ids": 10,
    "unique_positions": 10,
    "willingness_questions": 5,
    "capacity_questions": 5,
    "single_choice_questions": 10,
    "required_questions": 10,
}


# Fail explicitly if the persisted data does not satisfy the contract.
for field, expected_value in expected.items():
    actual_value = dq[field]

    if actual_value != expected_value:
        raise RuntimeError(f"DQ failure: {field}={actual_value}, expected {expected_value}")


print("DATA QUALITY CHECK: PASS")

for field, value in dq.items():
    print(f"{field}: {value}")

DATA QUALITY CHECK: PASS
total_questions: 10
unique_question_ids: 10
unique_positions: 10
willingness_questions: 5
capacity_questions: 5
single_choice_questions: 10
required_questions: 10


In [17]:
# Verify that all proposed scoring weights were persisted correctly.
# The v1 questionnaire contains 50 answer options distributed evenly
# across scores 1 through 5.

score_distribution = {}

for row in question_rows:
    for answer in row["answer_options"]:
        score = answer["proposed_score"]
        score_distribution[score] = score_distribution.get(score, 0) + 1


# Each of the five scoring weights should occur once per question,
# resulting in 10 occurrences for every score across 10 questions.
expected_distribution = {
    1: 10,
    2: 10,
    3: 10,
    4: 10,
    5: 10,
}


# Do not allow unexpected, missing, or duplicated scoring weights.
if score_distribution != expected_distribution:
    raise RuntimeError(f"Unexpected scoring distribution: {score_distribution}")


print("SCORING DQ CHECK: PASS")
print("Score distribution:", score_distribution)

SCORING DQ CHECK: PASS
Score distribution: {1: 10, 2: 10, 3: 10, 4: 10, 5: 10}
